In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

# -------------------------
# Load datasets
# -------------------------
train = pd.read_parquet('../data/model/final_train.parquet')
val   = pd.read_parquet('../data/model/final_val.parquet')
test  = pd.read_parquet('../data/model/final_test.parquet')

train = train.sort_values(["tic", "Date"])
val   = val.sort_values(["tic", "Date"])
test  = test.sort_values(["tic", "Date"])

# -------------------------
# Feature groups
# -------------------------
fund_features = [
    "sales_growth_qoq", "sales_growth_ttm", "asset_growth", "equity_growth",
    "roa_ttm", "roe_ttm", "gross_margin_ttm", "oper_margin_ttm",
    "net_margin_ttm", "log_mktcap", "bm", "earnings_yield", "cf_yield",
    "sales_yield", "div_yield", "leverage", "current_ratio", "cash_assets",
    "accruals_ta"
]

market_features = ["Open", "High", "Low", "Close", "Volume"]

macro_features = [
    "cpi", "fedfunds", "industrial_production", "gdp", "retail_sales",
    "unemployment", "t10y", "t2y", "t3m", "aaa_yield", "vix", "sp500",
    "yield_spread_10y_2y"
]

emb_cols = [c for c in train.columns if c.startswith("pca_emb_")]
news_features = ["mean_sentiment", "max_sentiment", "min_sentiment",
                 "sum_sentiment", "news_count"]

feature_groups = {
    "market": market_features,
    "fundamental": fund_features,
    "macro": macro_features,
    "news": news_features,
    "market+fund": market_features + fund_features,
    "market+macro": market_features + macro_features,
    "market+news": market_features + news_features,
    "ALL": market_features + fund_features + macro_features + news_features
}

# -------------------------
# Utility metrics
# -------------------------
def rmse(y_true, y_pred):
    mask = ~np.isnan(y_true)
    return np.sqrt(mean_squared_error(y_true[mask], y_pred[mask]))

def direction_accuracy(y_true, y_pred):
    mask = (~np.isnan(y_true))
    return (np.sign(y_true[mask]) == np.sign(y_pred[mask])).mean()

y_train = train["return_next_day"].values
y_val   = val["return_next_day"].values
y_test  = test["return_next_day"].values


In [9]:
train = train.dropna(subset=["return_next_day"])
val   = val.dropna(subset=["return_next_day"])
test  = test.dropna(subset=["return_next_day"])

y_train = train["return_next_day"].values
y_val   = val["return_next_day"].values
y_test  = test["return_next_day"].values


In [14]:
import xgboost as xgb

RANDOM_STATE = 42
import xgboost as xgb
from xgboost.callback import EarlyStopping
import xgboost as xgb

def train_xgb_gpu(feature_group_name, sample_frac=0.3):
    cols = feature_groups[feature_group_name]

    X_train = train[cols].values
    X_val   = val[cols].values
    X_test  = test[cols].values

    # Downsample for speed
    if sample_frac < 1.0:
        rng = np.random.RandomState(42)
        idx = rng.choice(len(X_train), int(len(X_train)*sample_frac), replace=False)
        X_train = X_train[idx]
        y_train_sub = y_train[idx]
    else:
        y_train_sub = y_train

    # Convert to DMatrix
    dtrain = xgb.DMatrix(X_train, label=y_train_sub)
    dval   = xgb.DMatrix(X_val,   label=y_val)
    dtest  = xgb.DMatrix(X_test,  label=y_test)

    # GPU XGBoost parameters
    params = {
        "objective": "reg:squarederror",
        "eval_metric": "rmse",
        "tree_method": "hist",
        "device": "cuda",
        "predictor": "gpu_predictor",
        "max_depth": 6,
        "eta": 0.05,
        "subsample": 0.7,
        "colsample_bytree": 0.7,
        "lambda": 1.0,
        "alpha": 0.0,
        "seed": 42,
    }

    evals = [(dtrain, "train"), (dval, "val")]

    # Train with early stopping (ALWAYS WORKS)
    model = xgb.train(
        params=params,
        dtrain=dtrain,
        num_boost_round=2000,
        evals=evals,
        early_stopping_rounds=50,
        verbose_eval=50,
    )

    # Predictions
    yhat_train = model.predict(dtrain)
    yhat_val   = model.predict(dval)
    yhat_test  = model.predict(dtest)

    print(f"\n==== XGBoost GPU — {feature_group_name} ====")
    print(f"Train RMSE: {rmse(y_train_sub, yhat_train):.6f}")
    print(f"Val   RMSE: {rmse(y_val,       yhat_val):.6f}")
    print(f"Test  RMSE: {rmse(y_test,      yhat_test):.6f}")
    print(f"Val   DA:   {direction_accuracy(y_val,  yhat_val):.4f}")
    print(f"Test  DA:   {direction_accuracy(y_test, yhat_test):.4f}")

    return model

# Run it:
model_all = train_xgb_gpu("ALL", sample_frac=0.3)





/home/zeyuzh/.conda/envs/jupyter_env/lib/python3.13/site-packages/xgboost/callback.py:386: UserWarning: [15:31:41] WARNING: /home/conda/feedstock_root/build_artifacts/xgboost-split_1762060556346/work/src/learner.cc:790: 
Parameters: { "predictor" } are not used.

  self.starting_round = model.num_boosted_rounds()


[0]	train-rmse:0.02115	val-rmse:0.02427
[50]	train-rmse:0.01875	val-rmse:0.02516

==== XGBoost GPU — ALL ====
Train RMSE: 0.018737
Val   RMSE: 0.025194
Test  RMSE: 0.021329
Val   DA:   0.5072
Test  DA:   0.4733


In [11]:
import xgboost as xgb
print("XGBoost version:", xgb.__version__)


XGBoost version: 3.1.1


In [15]:
import xgboost as xgb

print("Supports CUDA?:", xgb.core._has_cuda_support())


AttributeError: module 'xgboost.core' has no attribute '_has_cuda_support'

In [33]:
import pandas as pd
import numpy as np
import xgboost as xgb

# =========================
# 1. Load & sort data
# =========================
train = pd.read_parquet('../data/model/final_train.parquet')
val   = pd.read_parquet('../data/model/final_val.parquet')
test  = pd.read_parquet('../data/model/final_test.parquet')

train = train.sort_values(["tic", "Date"])
val   = val.sort_values(["tic", "Date"])
test  = test.sort_values(["tic", "Date"])

# =========================
# 2. Helper: market feature engineering
# =========================
def add_market_features(df):
    df = df.copy()
    df = df.sort_values(["tic", "Date"])

    # Daily return
    df["ret_1d"] = df.groupby("tic")["Close"].pct_change(1)

    # 5-day & 10-day returns (also used as momentum)
    df["ret_5d"] = df.groupby("tic")["Close"].pct_change(5)
    df["ret_10d"] = df.groupby("tic")["Close"].pct_change(10)

    # Rolling volatility of daily returns
    grp = df.groupby("tic")["ret_1d"]
    df["vol_5d"] = grp.rolling(5).std().reset_index(0, drop=True)
    df["vol_10d"] = grp.rolling(10).std().reset_index(0, drop=True)

    # Momentum aliases (for clarity)
    df["mom_5d"] = df["ret_5d"]
    df["mom_10d"] = df["ret_10d"]

    # Fill any NaNs in engineered features with 0
    for col in ["ret_1d", "ret_5d", "ret_10d", "vol_5d", "vol_10d", "mom_5d", "mom_10d"]:
        df[col] = df[col].fillna(0.0)

    return df

train = add_market_features(train)
val   = add_market_features(val)
test  = add_market_features(test)

# =========================
# 3. Define feature groups
# =========================

# Original groups for reference (if you still want to use them)
fund_features = [
    "sales_growth_qoq", "sales_growth_ttm", "asset_growth", "equity_growth",
    "roa_ttm", "roe_ttm", "gross_margin_ttm", "oper_margin_ttm",
    "net_margin_ttm", "log_mktcap", "bm", "earnings_yield", "cf_yield",
    "sales_yield", "div_yield", "leverage", "current_ratio", "cash_assets",
    "accruals_ta"
]

market_features_raw = ["Open", "High", "Low", "Close", "Volume"]

macro_features = [
    "cpi", "fedfunds", "industrial_production", "gdp", "retail_sales",
    "unemployment", "t10y", "t2y", "t3m", "aaa_yield", "vix", "sp500",
    "yield_spread_10y_2y"
]

news_features = ["mean_sentiment", "max_sentiment", "min_sentiment",
                 "sum_sentiment", "news_count"]

feature_groups = {}

feature_groups["ALL_original"] = (
    market_features_raw + fund_features + macro_features + news_features
)

# ---- Our improved, smaller set of features ----
improved_features = [
    # Market / technical
    "ret_1d", "ret_5d", "ret_10d",
    "vol_5d", "vol_10d", "mom_5d", "mom_10d",

    # News
    "mean_sentiment", "sum_sentiment", "news_count",

    # Key fundamentals
    "log_mktcap", "bm", "leverage", "roe_ttm", "roa_ttm",

    # Macro (small subset)
    "vix", "sp500", "yield_spread_10y_2y",
] 

feature_groups["improved"] = improved_features

# =========================
# 4. Utility metrics
# =========================
from sklearn.metrics import mean_squared_error

def rmse(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    mask = ~np.isnan(y_true)
    return np.sqrt(mean_squared_error(y_true[mask], y_pred[mask]))

def direction_accuracy(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    mask = ~np.isnan(y_true)
    return (np.sign(y_true[mask]) == np.sign(y_pred[mask])).mean()

# =========================
# 5. Clean target (drop NaNs)
# =========================
# We must drop rows where return_next_day is NaN (e.g., last day per ticker)
def drop_nan_target(df, target_col="return_next_day"):
    df = df.copy()
    return df[~df[target_col].isna()]

train = drop_nan_target(train)
val   = drop_nan_target(val)
test  = drop_nan_target(test)

y_train = train["return_next_day"].values
y_val   = val["return_next_day"].values
y_test  = test["return_next_day"].values

# =========================
# 6. XGBoost GPU trainer using DMatrix
# =========================
RANDOM_STATE = 42

def train_xgb_gpu(feature_group_name, sample_frac=0.5):
    cols = feature_groups[feature_group_name]

    # Design matrices
    X_train = train[cols].values
    X_val   = val[cols].values
    X_test  = test[cols].values

    # Optional downsampling of training set for speed
    if sample_frac < 1.0:
        rng = np.random.RandomState(RANDOM_STATE)
        n = X_train.shape[0]
        idx = rng.choice(n, int(n * sample_frac), replace=False)
        X_train_sub = X_train[idx]
        y_train_sub = y_train[idx]
    else:
        X_train_sub = X_train
        y_train_sub = y_train

    # DMatrix
    dtrain = xgb.DMatrix(X_train_sub, label=y_train_sub)
    dval   = xgb.DMatrix(X_val,       label=y_val)
    dtest  = xgb.DMatrix(X_test,      label=y_test)

    # XGBoost GPU params (hist + device=cuda)
    params = {
        "objective": "reg:squarederror",
        "eval_metric": "rmse",
        "tree_method": "hist",
        "device": "cuda",          # use GPU
        "max_depth": 6,
        "min_child_weight": 3,
        "eta": 0.02,
        "subsample": 1.0,
        "colsample_bytree": 0.6,
        "lambda": 3.0,
        "alpha": 0.0,
        "seed": RANDOM_STATE,
    }

    evals = [(dtrain, "train"), (dval, "val")]

    # Train with early stopping
    model = xgb.train(
        params=params,
        dtrain=dtrain,
        num_boost_round=2000,
        evals=evals,
        early_stopping_rounds=50,
        verbose_eval=50,
    )

    # Predictions
    yhat_train = model.predict(dtrain)
    yhat_val   = model.predict(dval)
    yhat_test  = model.predict(dtest)

    print(f"\n==== XGBoost (GPU) — Feature group: {feature_group_name} ====")
    print(f"Train RMSE: {rmse(y_train_sub, yhat_train):.6f}")
    print(f"Val   RMSE: {rmse(y_val,       yhat_val):.6f}")
    print(f"Test  RMSE: {rmse(y_test,      yhat_test):.6f}")
    print(f"Val   DA:   {direction_accuracy(y_val,  yhat_val):.4f}")
    print(f"Test  DA:   {direction_accuracy(y_test, yhat_test):.4f}")

    return model
# =========================
# 7. Run the improved model
# =========================
model_improved = train_xgb_gpu("improved", sample_frac=0.5)

# If you also want to compare with original ALL feature set:
# model_all_orig = train_xgb_gpu("ALL_original", sample_frac=0.3)


[0]	train-rmse:0.02121	val-rmse:0.02427
[50]	train-rmse:0.01987	val-rmse:0.02436
[58]	train-rmse:0.01976	val-rmse:0.02438

==== XGBoost (GPU) — Feature group: improved ====
Train RMSE: 0.019737
Val   RMSE: 0.024395
Test  RMSE: 0.018451
Val   DA:   0.4957
Test  DA:   0.4856


In [34]:
def add_return_5d(df):
    df = df.sort_values(["tic", "Date"]).copy()
    df["return_5d"] = df.groupby("tic")["Close"].pct_change(30).shift(-230)
    return df

train = add_return_5d(train)
val   = add_return_5d(val)
test  = add_return_5d(test)

# drop rows where we cannot compute 5-day return
train = train.dropna(subset=["return_5d"])
val   = val.dropna(subset=["return_5d"])
test  = test.dropna(subset=["return_5d"])

y_train = train["return_5d"].values
y_val   = val["return_5d"].values
y_test  = test["return_5d"].values


In [35]:
import xgboost as xgb

def train_xgb_gpu(feature_group_name, sample_frac=0.5):
    cols = feature_groups[feature_group_name]

    X_train = train[cols].values
    X_val   = val[cols].values
    X_test  = test[cols].values

    # downsample to speed up training
    if sample_frac < 1.0:
        rng = np.random.RandomState(42)
        idx = rng.choice(len(X_train), int(len(X_train)*sample_frac), replace=False)
        X_train_sub = X_train[idx]
        y_train_sub = y_train[idx]
    else:
        X_train_sub = X_train
        y_train_sub = y_train

    dtrain = xgb.DMatrix(X_train_sub, label=y_train_sub)
    dval   = xgb.DMatrix(X_val,       label=y_val)
    dtest  = xgb.DMatrix(X_test,      label=y_test)

    params = {
        "objective": "reg:squarederror",
        "eval_metric": "rmse",
        "tree_method": "hist",
        "device": "cuda",
        "max_depth": 8,
        "min_child_weight": 3,
        "eta": 0.03,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "lambda": 2.0,
        "alpha": 0.0,
        "seed": 42,
    }

    model = xgb.train(
        params=params,
        dtrain=dtrain,
        num_boost_round=2000,
        evals=[(dtrain, "train"), (dval, "val")],
        early_stopping_rounds=50,
        verbose_eval=50,
    )

    yhat_train = model.predict(dtrain)
    yhat_val   = model.predict(dval)
    yhat_test  = model.predict(dtest)

    print(f"\n==== XGBoost GPU (5-day return) — {feature_group_name} ====")
    print(f"Train RMSE: {rmse(y_train_sub, yhat_train):.6f}")
    print(f"Val   RMSE: {rmse(y_val,       yhat_val):.6f}")
    print(f"Test  RMSE: {rmse(y_test,      yhat_test):.6f}")
    print(f"Val   DA:   {direction_accuracy(y_val,  yhat_val):.4f}")
    print(f"Test  DA:   {direction_accuracy(y_test, yhat_test):.4f}")

    return model

# Run it on improved features
model_5d = train_xgb_gpu("improved", sample_frac=0.5)


[0]	train-rmse:0.10931	val-rmse:0.12727
[49]	train-rmse:0.09415	val-rmse:0.13329

==== XGBoost GPU (5-day return) — improved ====
Train RMSE: 0.094030
Val   RMSE: 0.133323
Test  RMSE: 0.116199
Val   DA:   0.4679
Test  DA:   0.5291


In [36]:
import numpy as np
import pandas as pd
from scipy.stats import spearmanr, pearsonr

# ============================================
# 1. BASIC METRICS
# ============================================

def rmse(y_true, y_pred):
    mask = ~np.isnan(y_true)
    return np.sqrt(np.mean((y_true[mask] - y_pred[mask])**2))

def directional_accuracy(y_true, y_pred):
    mask = ~np.isnan(y_true)
    return np.mean(np.sign(y_true[mask]) == np.sign(y_pred[mask]))

def information_coefficient(y_true, y_pred):
    mask = ~np.isnan(y_true)
    pear = pearsonr(y_pred[mask], y_true[mask])[0]
    spear = spearmanr(y_pred[mask], y_true[mask])[0]
    return pear, spear

def cross_sectional_r2(y_true, y_pred):
    mask = ~np.isnan(y_true)
    return 1 - np.var(y_true[mask] - y_pred[mask]) / np.var(y_true[mask])


# ============================================
# 2. DECILE / QUANTILE ANALYSIS
# ============================================

def quantile_analysis(df, pred_col, ret_col="return_30d", n=10):
    df = df[[pred_col, ret_col]].dropna()
    df["quantile"] = pd.qcut(df[pred_col], n, labels=False)

    quantile_returns = df.groupby("quantile")[ret_col].mean()
    return quantile_returns


# ============================================
# 3. LONG-SHORT PORTFOLIO BACKTEST
# ============================================

def long_short_portfolio(df, pred_col, ret_col="return_30d", top=0.2):
    df = df[[pred_col, ret_col]].dropna()

    cutoff_top = df[pred_col].quantile(1 - top)
    cutoff_bot = df[pred_col].quantile(top)

    long = df[df[pred_col] >= cutoff_top][ret_col].mean()
    short = df[df[pred_col] <= cutoff_bot][ret_col].mean()

    long_short_return = long - short
    return long_short_return, long, short


# ============================================
# 4. SHARPE RATIO
# ============================================

def sharpe_ratio(returns, freq=12):  
    # freq=12 for monthly, 252 for daily
    return returns.mean() / returns.std()


# ============================================
# 5. ROLLING-WINDOW METRICS
# ============================================

def rolling_performance(df, pred_col, ret_col="return_30d", window=63):
    df = df[[pred_col, ret_col]].dropna().copy()

    roll_da = df.apply(lambda row: np.sign(row[pred_col]) == np.sign(row[ret_col]), axis=1)
    roll_da = roll_da.rolling(window).mean()

    roll_ic = df[[pred_col, ret_col]].rolling(window).corr().unstack().iloc[:,1]

    return roll_da, roll_ic


# ============================================
# 6. FULL EVALUATION FUNCTION
# ============================================

def evaluate_model(df, pred_col, ret_col="return_30d"):
    y_true = df[ret_col].values
    y_pred = df[pred_col].values

    print("\n====== BASIC METRICS ======")
    print("RMSE:", rmse(y_true, y_pred))
    print("Directional Accuracy:", directional_accuracy(y_true, y_pred))
    pear, spear = information_coefficient(y_true, y_pred)
    print("Pearson IC:", pear)
    print("Spearman IC:", spear)
    print("Cross-sectional R²:", cross_sectional_r2(y_true, y_pred))

    print("\n====== DECILE ANALYSIS ======")
    q = quantile_analysis(df, pred_col, ret_col)
    print(q)

    print("\n====== LONG-SHORT PORTFOLIO ======")
    ls, long_ret, short_ret = long_short_portfolio(df, pred_col, ret_col)
    print("Long-Short Return:", ls)
    print("Long Portfolio Return:", long_ret)
    print("Short Portfolio Return:", short_ret)

    return {
        "RMSE": rmse(y_true, y_pred),
        "DA": directional_accuracy(y_true, y_pred),
        "Pearson IC": pear,
        "Spearman IC": spear,
        "CS R2": cross_sectional_r2(y_true, y_pred),
        "Deciles": q,
        "LongShort": ls,
        "Long": long_ret,
        "Short": short_ret,
    }


In [39]:
test_df["pred"] = model_5d.predict(improved_features)
test_df["return_30d"] = y_test


TypeError: ('Expecting data to be a DMatrix object, got: ', <class 'list'>)

In [31]:
import xgboost as xgb
import numpy as np

# Define the parameter grid (moderately sized)
param_grid = {
    "max_depth":            [4, 6, 8, 10],
    "min_child_weight":     [1, 3, 5],
    "eta":                  [0.02, 0.03, 0.05, 0.1],
    "subsample":            [0.6, 0.8, 1.0],
    "colsample_bytree":     [0.6, 0.8, 1.0],
    "lambda":               [1.0, 3.0, 5.0],
    "alpha":                [0.0, 0.1],
}

# Convert to DMatrix
dtrain = xgb.DMatrix(train[improved_features].values, label=y_train)
dval   = xgb.DMatrix(val[improved_features].values,   label=y_val)
dtest  = xgb.DMatrix(test[improved_features].values,  label=y_test)

def train_model(params):
    """Train one model using xgb.train with early stopping."""
    params_full = {
        "objective": "reg:squarederror",
        "eval_metric": "rmse",
        "tree_method": "hist",
        "device": "cuda",
        "seed": 42,
    }
    params_full.update(params)

    evals = [(dtrain, "train"), (dval, "val")]

    model = xgb.train(
        params=params_full,
        dtrain=dtrain,
        num_boost_round=2000,
        evals=evals,
        early_stopping_rounds=50,
        verbose_eval=False,
    )
    return model

# ------------------------------
# Perform the tuning loop
# ------------------------------
import itertools

best_rmse = float("inf")
best_params = None
best_model = None

for max_depth, min_child_weight, eta, subsample, colsample_bytree, lam, alp in itertools.product(
    param_grid["max_depth"],
    param_grid["min_child_weight"],
    param_grid["eta"],
    param_grid["subsample"],
    param_grid["colsample_bytree"],
    param_grid["lambda"],
    param_grid["alpha"],
):
    params = {
        "max_depth": max_depth,
        "min_child_weight": min_child_weight,
        "eta": eta,
        "subsample": subsample,
        "colsample_bytree": colsample_bytree,
        "lambda": lam,
        "alpha": alp,
    }

    model = train_model(params)
    val_pred = model.predict(dval)

    # compute RMSE on validation
    val_rmse = np.sqrt(np.mean((y_val - val_pred)**2))

    print(f"Params: {params} → Val RMSE: {val_rmse:.5f}")

    if val_rmse < best_rmse:
        best_rmse = val_rmse
        best_params = params
        best_model = model

print("\n==================== BEST MODEL ====================")
print("Best Params:", best_params)
print("Best Val RMSE:", best_rmse)

# Final Test Performance
test_pred = best_model.predict(dtest)
test_rmse = np.sqrt(np.mean((y_test - test_pred)**2))

print("Test RMSE:", test_rmse)

# Directional Accuracy
def DA(y_true, y_pred):
    mask = ~np.isnan(y_true)
    return (np.sign(y_true[mask]) == np.sign(y_pred[mask])).mean()

print("Test DA:", DA(y_test, test_pred))


Params: {'max_depth': 4, 'min_child_weight': 1, 'eta': 0.02, 'subsample': 0.6, 'colsample_bytree': 0.6, 'lambda': 1.0, 'alpha': 0.0} → Val RMSE: 0.12904
Params: {'max_depth': 4, 'min_child_weight': 1, 'eta': 0.02, 'subsample': 0.6, 'colsample_bytree': 0.6, 'lambda': 1.0, 'alpha': 0.1} → Val RMSE: 0.12905
Params: {'max_depth': 4, 'min_child_weight': 1, 'eta': 0.02, 'subsample': 0.6, 'colsample_bytree': 0.6, 'lambda': 3.0, 'alpha': 0.0} → Val RMSE: 0.12905
Params: {'max_depth': 4, 'min_child_weight': 1, 'eta': 0.02, 'subsample': 0.6, 'colsample_bytree': 0.6, 'lambda': 3.0, 'alpha': 0.1} → Val RMSE: 0.12905
Params: {'max_depth': 4, 'min_child_weight': 1, 'eta': 0.02, 'subsample': 0.6, 'colsample_bytree': 0.6, 'lambda': 5.0, 'alpha': 0.0} → Val RMSE: 0.12905
Params: {'max_depth': 4, 'min_child_weight': 1, 'eta': 0.02, 'subsample': 0.6, 'colsample_bytree': 0.6, 'lambda': 5.0, 'alpha': 0.1} → Val RMSE: 0.12905
Params: {'max_depth': 4, 'min_child_weight': 1, 'eta': 0.02, 'subsample': 0.6, 'co

In [32]:
# save the best model
best_model.save_model('./data/train_model/xgb_best_model.json')

XGBoostError: [16:36:35] /home/conda/feedstock_root/build_artifacts/xgboost-split_1762060556346/work/dmlc-core/src/io/local_filesys.cc:210: Check failed: allow_null:  LocalFileSystem::Open "./data/train_model/xgb_best_model.json": No such file or directory
Stack trace:
  [bt] (0) /home/zeyuzh/.conda/envs/jupyter_env/lib/libxgboost.so(dmlc::LogMessageFatal::~LogMessageFatal()+0x6e) [0x14b5c0b3b96e]
  [bt] (1) /home/zeyuzh/.conda/envs/jupyter_env/lib/libxgboost.so(dmlc::io::LocalFileSystem::Open(dmlc::io::URI const&, char const*, bool)+0x235) [0x14b5c2022905]
  [bt] (2) /home/zeyuzh/.conda/envs/jupyter_env/lib/libxgboost.so(dmlc::Stream::Create(char const*, char const*, bool)+0x29a) [0x14b5c200a09a]
  [bt] (3) /home/zeyuzh/.conda/envs/jupyter_env/lib/libxgboost.so(XGBoosterSaveModel+0x4f) [0x14b5c0a82b6f]
  [bt] (4) /home/zeyuzh/.conda/envs/jupyter_env/lib/python3.13/lib-dynload/../../libffi.so.8(+0x702a) [0x14b6964bc02a]
  [bt] (5) /home/zeyuzh/.conda/envs/jupyter_env/lib/python3.13/lib